# AH | Gráficos de exploración y control

**Autor:** Alan Arturo Hernández Gutiérrez.

Prepara figuras a partir de los agregados de exploración y de las salidas validadas del modelo.

Este cuaderno define las operaciones que utilizan los cuadernos de `notebooks/`. Ejecute sus celdas en orden. El flujo completo se inicia desde 01 o 02.

## Raíz del proyecto

La ubicación se resuelve desde la raíz, `src/` o `notebooks/`, sin guardar rutas personales.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name.lower() in {'src', 'notebooks'}:
    ROOT = ROOT.parent
if not (ROOT / 'config' / 'proyecto.json').is_file():
    raise RuntimeError('Inicie Jupyter desde la raíz del proyecto, src/ o notebooks/.')


## Bibliotecas y parámetros

Se preparan las bibliotecas y constantes utilizadas por las operaciones siguientes.

In [ ]:
from pathlib import Path

import json

import numpy as np

import pandas as pd

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt


## Guardar una figura de evidencia

Ajusta los márgenes y escribe la figura en evidencia/graficos/; devuelve su ruta y cierra la figura para liberar memoria.

In [ ]:
def save(fig, name):
    dest = ROOT / "evidencia" / "graficos"
    dest.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(str(dest / (name + ".png")), dpi=150, facecolor="white")
    plt.close(fig)
    return dest / (name + ".png")


## Exploratory

Genera las figuras del cuaderno exploratorio a partir de sus agregados.

In [ ]:
def exploratory():
    p = json.loads((ROOT / "evidencia" / "perfil_fuentes.json").read_text(encoding="utf-8"))
    d = json.loads(
        (ROOT / "evidencia" / "diagnosticos_adicionales.json").read_text(encoding="utf-8")
    )
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 10,
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )
    out = []
    monthly = pd.Series(p["transactions"]["rows_by_month"])
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.bar(np.arange(len(monthly)), monthly.values, color="#248CDD")
    ax.set_xticks(np.arange(len(monthly)))
    ax.set_xticklabels(monthly.index, rotation=60, ha="right")
    ax.set_title("Líneas registradas por mes", loc="left", color="#12263E", fontweight="bold")
    ax.set_ylabel("Líneas de movimientos")
    ax.grid(axis="y", alpha=0.2)
    out.append(save(fig, "eda_01_cobertura_temporal"))
    cat = pd.DataFrame(d["negative_amounts"]["by_category"]).dropna(subset=["CATEGORIA_NOMBRE"])
    cat["pct"] = 100 * cat.negative_rows / cat.rows
    cat = cat.sort_values("pct")
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.barh(cat.CATEGORIA_NOMBRE, cat.pct, color="#174A79")
    ax.set_xlabel("% de líneas con monto negativo dentro de cada categoría")
    ax.set_title(
        "El signo del importe requiere una regla explícita",
        loc="left",
        color="#12263E",
        fontweight="bold",
    )
    ax.set_xlim(0, max(cat.pct) * 1.2)
    for i, val in enumerate(cat.pct):
        ax.text(val + 0.3, i, "{:.1f}%".format(val), va="center")
    out.append(save(fig, "eda_02_importes_negativos"))
    a = d["active_customers"]
    labels = ["Edad ausente", "Domicilio municipal ausente", "Segmento no encontrado"]
    vals = [100 * a[k] / a["customers"] for k in ["age_missing", "home_missing", "segment_missing"]]
    fig, ax = plt.subplots(figsize=(9, 3.8))
    ax.barh(labels, vals, color="#248CDD")
    ax.set_xlim(0, 100)
    ax.set_xlabel("% de clientes con movimientos")
    ax.set_title(
        "Cobertura de atributos del cliente", loc="left", color="#12263E", fontweight="bold"
    )
    for i, val in enumerate(vals):
        ax.text(val + 0.4, i, "{:.2f}%".format(val), va="center")
    out.append(save(fig, "eda_03_cobertura_cliente"))
    return out


## Model results

Genera las figuras de cohortes y sensibilidad a partir del modelo validado.

In [ ]:
def model_results(current):
    current = Path(current)
    cohorts = pd.read_csv(current / "analisis" / "Cohortes90.csv")
    cohorts = cohorts.loc[cohorts.ReglaKey.eq(1) & cohorts.Elegibles90.gt(0)]
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot(cohorts.CohorteMes, 100 * cohorts.TasaAdopcion90, marker="o", color="#174A79")
    for i, row in enumerate(cohorts.itertuples(index=False)):
        ax.annotate(
            "n={:,}".format(int(row.Elegibles90)),
            (i, 100 * row.TasaAdopcion90),
            textcoords="offset points",
            xytext=(0, 9),
            ha="center",
            fontsize=8,
        )
    ax.set_ylim(-2, 115)
    ax.tick_params(axis="x", rotation=60)
    ax.set_ylabel("% de elegibles con adopción observada a 90 días")
    ax.set_title(
        "Cohortes maduras: regla base observada", loc="left", color="#12263E", fontweight="bold"
    )
    ax.grid(axis="y", alpha=0.2)
    fig.text(
        0.02,
        0.01,
        "n = clientes elegibles. Abril 2020 tiene solo 2; no extrapolar su 100%. Primera compra observada, no adquisición real.",
        fontsize=8,
    )
    out = [save(fig, "etl_01_cohortes90")]
    s = pd.read_csv(current / "analisis" / "SensibilidadReglas.csv")
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(
        s.ReglaNombre, 100 * s.TasaAdopcion90, color=["#12263E", "#174A79", "#248CDD", "#739EC6"]
    )
    ax.set_ylabel("% de adopción observada a 90 días")
    ax.set_title(
        "Sensibilidad a la definición de movimiento", loc="left", color="#12263E", fontweight="bold"
    )
    for i, val in enumerate(s.TasaAdopcion90):
        ax.text(i, val * 100 + 0.15, "{:.2f}%".format(val * 100), ha="center")
    out.append(save(fig, "etl_02_sensibilidad"))
    return out
